# 00 Colab 環境設定

這個 notebook 負責：掛載 Google Drive、安裝相依套件、把專案程式碼放進 Colab、檢查 GPU 與版本。

**前置作業**：把本機的專案資料夾壓縮上傳到 `MyDrive/tetrio-ai/code/tetrio-ai.zip`，或改成 `git clone`。

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/tetrio-ai'
os.environ['TETRIO_AI_DRIVE'] = DRIVE_ROOT
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

In [ ]:
# 取得程式碼：預設直接從 GitHub 抓（公開 repo，不需要登入）
REPO_URL = 'https://github.com/wallacechen0130/tetr_bot.git'
PROJECT = '/content/tetrio-ai'
ZIP_PATH = os.path.join(DRIVE_ROOT, 'code', 'tetrio-ai.zip')  # 備援：Drive 上的 zip

import subprocess
import zipfile

if os.path.isdir(os.path.join(PROJECT, '.git')):
    print('已有 checkout，改成更新：git pull')
    subprocess.run(f'git -C {PROJECT} pull --ff-only', shell=True)
elif os.path.exists(os.path.join(PROJECT, 'requirements.txt')):
    print('已有專案目錄，略過下載：', PROJECT)
elif os.path.exists(ZIP_PATH):
    print('由 Drive zip 解壓：', ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall('/content')
else:
    print('git clone：', REPO_URL)
    result = subprocess.run(f'git clone --depth 1 {REPO_URL} {PROJECT}', shell=True)
    if result.returncode != 0:
        raise SystemExit(
            'git clone 失敗。請確認網路，或把專案 zip 上傳到 ' + ZIP_PATH
        )

os.chdir(PROJECT if os.path.isdir(PROJECT) else '/content')
missing = [name for name in ('requirements.txt', 'envs/gym/tetris_env.py', 'scripts/run_smoke.py')
           if not os.path.exists(name)]
if missing:
    raise SystemExit('專案內容不完整，缺少：' + ', '.join(missing))
print('工作目錄:', os.getcwd())
print(subprocess.run('git log --oneline -1', shell=True, capture_output=True, text=True).stdout.strip())

In [ ]:
%pip install -q -r requirements-colab.txt

In [ ]:
import sys

import torch

print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
else:
    print('沒有 GPU，建議只做資料驗證，訓練請開 GPU runtime')

In [ ]:
# 快速驗證：跑最小測試與 smoke（不訓練 PPO）
!python -m pytest tests -q -x --ignore=tests/test_smoke.py
!python -m scripts.run_smoke --skip-ppo